In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv('ft_eng_data_v3.csv')
df.head()

,customer_id,name,signup_date,last_active_date,base_salary,overtime_pay,bonus,investment_income,total_annual_income,monthly_spend,support_tickets,referrals,balance_adjustment,city,plan
0,1001,Amara Khan,2022-01-02,2022-03-08,73300.0,0.00,0.00,0.00,73300.00,13.45,1,0,-512.54,Lagos,basic
1,1002,Dev Haddad,2023-13-45,NaN,101500.0,0.00,0.00,3022.45,104522.45,15.28,4,0,-53.49,Berlin,basic
2,1003,NaN,NaN,2022-01-11,88800.0,0.00,1649.74,0.00,90449.74,16.09,0,0,-2781.59,Berlin,basic
3,1004,Lena Costa,2022-01-03,2024-10-01,10100.0,0.00,10734.62,0.00,20834.62,11.22,2,0,1823.98,Berlin,premium
4,1005,Kwame Haddad,2022-01-03,2022-02-04,93800.0,5138.04,6202.98,7876.21,NaN,15.31,0,0,5504.51,Lima,basic


In [16]:
print(df.isna().sum())

customer_id             0
name                   11
signup_date            12
last_active_date       29
base_salary             0
overtime_pay            0
bonus                   0
investment_income      24
total_annual_income    65
monthly_spend          10
support_tickets         0
referrals               0
balance_adjustment      0
city                    9
plan                   15
dtype: int64


In [17]:
df["signup_date"] = pd.to_datetime(df.signup_date, errors="coerce")
df["last_active_date"] = pd.to_datetime(df.last_active_date, errors="coerce")
df.head()

,customer_id,name,signup_date,last_active_date,base_salary,overtime_pay,bonus,investment_income,total_annual_income,monthly_spend,support_tickets,referrals,balance_adjustment,city,plan
0,1001,Amara Khan,2022-01-02,2022-03-08,73300.0,0.00,0.00,0.00,73300.00,13.45,1,0,-512.54,Lagos,basic
1,1002,Dev Haddad,NaT,NaT,101500.0,0.00,0.00,3022.45,104522.45,15.28,4,0,-53.49,Berlin,basic
2,1003,NaN,NaT,2022-01-11,88800.0,0.00,1649.74,0.00,90449.74,16.09,0,0,-2781.59,Berlin,basic
3,1004,Lena Costa,2022-01-03,2024-10-01,10100.0,0.00,10734.62,0.00,20834.62,11.22,2,0,1823.98,Berlin,premium
4,1005,Kwame Haddad,2022-01-03,2022-02-04,93800.0,5138.04,6202.98,7876.21,NaN,15.31,0,0,5504.51,Lima,basic


In [18]:
MONEY = ["base_salary", "overtime_pay", "bonus", "investment_income", "total_annual_income", "monthly_spend"
        , "balance_adjustment"]

for c in MONEY:
    df[c] = pd.to_numeric(df[c], errors="coerce")

n_sent = (df.monthly_spend == -1.0).sum()
df.loc[df.monthly_spend ==-1.0, "monthly_spend"] = np.nan
df.head()

,customer_id,name,signup_date,last_active_date,base_salary,overtime_pay,bonus,investment_income,total_annual_income,monthly_spend,support_tickets,referrals,balance_adjustment,city,plan
0,1001,Amara Khan,2022-01-02,2022-03-08,73300.0,0.00,0.00,0.00,73300.00,13.45,1,0,-512.54,Lagos,basic
1,1002,Dev Haddad,NaT,NaT,101500.0,0.00,0.00,3022.45,104522.45,15.28,4,0,-53.49,Berlin,basic
2,1003,NaN,NaT,2022-01-11,88800.0,0.00,1649.74,0.00,90449.74,16.09,0,0,-2781.59,Berlin,basic
3,1004,Lena Costa,2022-01-03,2024-10-01,10100.0,0.00,10734.62,0.00,20834.62,11.22,2,0,1823.98,Berlin,premium
4,1005,Kwame Haddad,2022-01-03,2022-02-04,93800.0,5138.04,6202.98,7876.21,NaN,15.31,0,0,5504.51,Lima,basic


In [19]:
print(n_sent)

8


In [22]:
impossible = df.last_active_date < df.signup_date
print(impossible.sum())

11


In [23]:
df["date_order_bad"] = impossible.astype(int)
df.loc[impossible, "last_active_date"] = pd.NaT

In [24]:
BLANKS = {"", "nan", "none", "null", "unknown", "n/a", "na", "--", "-", "?", "."}

def to_null(s):
    def one(v):
        if pd.isna(v):
            return np.nan
        t = str(v).strip()
        return np.nan if t.lower() in BLANKS else t
    return s.astype("object").map(one)

In [29]:
df["city"] = to_null(df["city"]).str.title().fillna("Unknown")
df["plan"] = to_null(df["plan"]).str.lower()
df["plan"] = df["plan"].fillna(df["plan"].mode()[0])
df["name"] = to_null(df["name"])

print(sorted(df.city.unique()))
print(sorted(df.plan.unique()))

['Berlin', 'Lagos', 'Lima', 'Osaka', 'Toronto', 'Unknown']
['basic', 'plus', 'premium']


In [31]:
df["income_missing"] = df["total_annual_income"].isna().astype(int)
df["date_missing"] = df["signup_date"].isna().astype(int)
df["never_active"] = df["last_active_date"].isna().astype(int)
df["invest_missing"] = df["investment_income"].isna().astype(int)

In [33]:
df.head()

,customer_id,name,signup_date,last_active_date,base_salary,overtime_pay,bonus,investment_income,total_annual_income,monthly_spend,support_tickets,referrals,balance_adjustment,city,plan,date_order_bad,income_missing,date_missing,never_active,invest_missing
0,1001,Amara Khan,2022-01-02,2022-03-08,73300.0,0.00,0.00,0.00,73300.00,13.45,1,0,-512.54,Lagos,basic,0,0,0,0,0
1,1002,Dev Haddad,NaT,NaT,101500.0,0.00,0.00,3022.45,104522.45,15.28,4,0,-53.49,Berlin,basic,0,0,1,1,0
2,1003,NaN,NaT,2022-01-11,88800.0,0.00,1649.74,0.00,90449.74,16.09,0,0,-2781.59,Berlin,basic,0,0,1,0,0
3,1004,Lena Costa,2022-01-03,2024-10-01,10100.0,0.00,10734.62,0.00,20834.62,11.22,2,0,1823.98,Berlin,premium,0,0,0,0,0
4,1005,Kwame Haddad,2022-01-03,2022-02-04,93800.0,5138.04,6202.98,7876.21,NaN,15.31,0,0,5504.51,Lima,basic,0,1,0,0,0


In [34]:
df["monthly_spend"] = df.groupby("city")["monthly_spend"].transform(lambda s: s.fillna(s.median()))
df["monthly_spend"] = df["monthly_spend"].fillna(df["monthly_spend"].median())

In [35]:
print()

0      13.45
1      15.28
2      16.09
3      11.22
4      15.31
       ...  
595    12.88
596    11.42
597    18.55
598    12.35
599    16.14
Name: monthly_spend, Length: 600, dtype: float64
